In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# ─────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn - Copy.csv")

print("Columns in your dataset:")
print(df.columns.tolist())
print("\nShape:", df.shape)

# ─────────────────────────────────────────
# 2. CLEAN DATA
# ─────────────────────────────────────────
# Drop customerID if it exists
if "customerID" in df.columns:
    df.drop(columns=["customerID"], inplace=True)

# Fix TotalCharges if it exists
if "TotalCharges" in df.columns:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop rows with missing values
df.dropna(inplace=True)

# Encode target: Yes → 1, No → 0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# ─────────────────────────────────────────
# 3. SPLIT FEATURES AND TARGET
# ─────────────────────────────────────────
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Auto-detect column types
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("\nNumerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

# ─────────────────────────────────────────
# 4. TRAIN / TEST SPLIT
# ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ─────────────────────────────────────────
# 5. PREPROCESSING
# ─────────────────────────────────────────
transformers = []
if numerical_cols:
    transformers.append(("num", StandardScaler(), numerical_cols))
if categorical_cols:
    transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols))

preprocessor = ColumnTransformer(transformers=transformers)

# ─────────────────────────────────────────
# 6. PIPELINES
# ─────────────────────────────────────────
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# ─────────────────────────────────────────
# 7. GRIDSEARCHCV - LOGISTIC REGRESSION
# ─────────────────────────────────────────
lr_params = {
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__solver": ["lbfgs", "liblinear"]
}

print("\nTuning Logistic Regression...")
lr_grid = GridSearchCV(lr_pipeline, lr_params, cv=5, scoring="accuracy", n_jobs=-1)
lr_grid.fit(X_train, y_train)
print("Best LR Params:", lr_grid.best_params_)
print("Best LR CV Score:", round(lr_grid.best_score_, 4))

# ─────────────────────────────────────────
# 8. GRIDSEARCHCV - RANDOM FOREST
# ─────────────────────────────────────────
rf_params = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 5, 10]
}

print("\nTuning Random Forest...")
rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=5, scoring="accuracy", n_jobs=-1)
rf_grid.fit(X_train, y_train)
print("Best RF Params:", rf_grid.best_params_)
print("Best RF CV Score:", round(rf_grid.best_score_, 4))

# ─────────────────────────────────────────
# 9. EVALUATE BOTH MODELS
# ─────────────────────────────────────────
print("\n===== Logistic Regression =====")
lr_preds = lr_grid.best_estimator_.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, lr_preds), 4))
print(classification_report(y_test, lr_preds))

print("\n===== Random Forest =====")
rf_preds = rf_grid.best_estimator_.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, rf_preds), 4))
print(classification_report(y_test, rf_preds))

# ─────────────────────────────────────────
# 10. PICK BEST MODEL & EXPORT
# ─────────────────────────────────────────
if lr_grid.best_score_ >= rf_grid.best_score_:
    best_model = lr_grid.best_estimator_
    print("\nBest Model: Logistic Regression")
else:
    best_model = rf_grid.best_estimator_
    print("\nBest Model: Random Forest")

joblib.dump(best_model, "churn_pipeline.joblib")
print("Pipeline saved to churn_pipeline.joblib")

# ─────────────────────────────────────────
# 11. LOAD & PREDICT (example)
# ─────────────────────────────────────────
loaded_model = joblib.load("churn_pipeline.joblib")
sample = X_test.iloc[:5]
predictions = loaded_model.predict(sample)
print("\nSample Predictions:", predictions)

Columns in your dataset:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Shape: (7043, 21)

Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Tuning Logistic Regression...


C:\Users\DELL\AppData\Local\Temp\ipykernel_24192\1809261489.py:46: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


Best LR Params: {'classifier__C': 10, 'classifier__solver': 'lbfgs'}
Best LR CV Score: 0.8073

Tuning Random Forest...
Best RF Params: {'classifier__max_depth': 10, 'classifier__n_estimators': 100}
Best RF CV Score: 0.8037

===== Logistic Regression =====
Accuracy: 0.7882
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.62      0.52      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407


===== Random Forest =====
Accuracy: 0.796
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1033
           1       0.66      0.49      0.56       374

    accuracy                           0.80      1407
   macro avg       0.74      0.70      0.71      1407
weighted avg       0.78      0.80      0.79      1407


Best Model: Logistic Regressi